# Precipitation Nowcasting — Colab conversion

Este notebook es la conversión del script `Backend/model/train_precipitation.py` para ejecutarlo en Google Colab. Sigue las celdas en orden: monta Drive (opcional), instala dependencias, carga datos, define funciones y modelos, y ejecuta el entrenamiento y evaluación.

Resumen:
- Predicción de lluvia intensa en horizontes 3h/6h/12h usando RNN/LSTM/GRU (y opcionalmente LNN)
- Datos: series de 15-min desde estaciones meteorológicas
- Salidas: checkpoints, curvas de entrenamiento, métricas y gráficas

Instrucciones rápidas:
1. Ejecuta la celda "Mount Google Drive" si quieres acceder a archivos en Drive.
2. Ejecuta la celda de instalación de dependencias (puede tardar).
3. Ajusta la celda de parámetros (DATA_DIR, OUTPUT_DIR, etc.).
4. Ejecuta las celdas en orden.


In [75]:
# Cell 2: Mount Google Drive (optional)
# Run this cell if you want to use files stored in Google Drive.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted at /content/drive')
except Exception:
    print('Not running in Colab or google.colab not available. Ignore if running locally.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted at /content/drive


## Install & Import Dependencies

Run the following cell to install required packages in Colab. This may take a few minutes. If you are running locally, you can skip installations you already have.

The next cell will run pip installs and then import the main packages used by the notebook.

In [76]:
# Cell 4: Install dependencies (Colab)
# Run this cell once in Colab. Adjust packages if needed.
import sys

if 'google.colab' in sys.modules:
    # Colab: install packages
    !pip install -q torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu118 || true
    !pip install -q pandas numpy matplotlib seaborn scikit-learn imbalanced-learn tqdm nbformat
    !pip install -q ncps || true
else:
    print('Not running in Colab; skip pip installs if running locally')

# Ensure seaborn is available and apply its style
try:
    import seaborn as sns
    sns.set()
except Exception:
    print('seaborn not available yet; if running in Colab re-run this cell after installs')

# Basic imports
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from datetime import timedelta

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import precision_recall_curve, average_precision_score, roc_auc_score, confusion_matrix, brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression

%matplotlib inline
warnings.filterwarnings('ignore')

# Print environment info
print('='*60)
print('Environment Configuration')
print('='*60)
print(f'Python: {sys.version}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  Device: {torch.cuda.get_device_name(0)}')
    print(f'  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'NumPy: {np.__version__}')
print(f'Pandas: {pd.__version__}')
print('='*60)
print('Environment ready!')


Environment Configuration
Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
PyTorch: 2.10.0+cpu
CUDA Available: False
NumPy: 2.0.2
Pandas: 2.2.2
Environment ready!


## 0. Setup & Data Download (R2)

Las siguientes celdas configuran las credenciales R2 y descargan el dataset "precipitation-nowcasting" desde el almacenamiento en la nube usado en el hackathon. Ejecuta estas celdas en Colab (o local si tienes las credenciales).

In [77]:
# Cell: R2 helper download + credentials + manifest download
!pip install -q boto3 tqdm netCDF4 || true

import os
import json
from pathlib import Path

# === Get r2_download.py helper module (if running on Colab/Kaggle) ===
HELPER_URL = "https://raw.githubusercontent.com/SALA-AI-LATAM/hackathon-participants/main/r2_download.py"

if not Path("r2_download.py").exists():
    print("Downloading r2_download.py...")
    import urllib.request
    urllib.request.urlretrieve(HELPER_URL, "r2_download.py")
    print("Done.")
else:
    print("r2_download.py already present.")

# === Set R2 credentials (example hackathon credentials) ===
os.environ["R2_ENDPOINT"] = "https://6200702e94592ad231a53daba00f8a5d.r2.cloudflarestorage.com"
os.environ["R2_ACCESS_KEY_ID"] = "93bb95ebfe47d5ef93c45efe3c108ca8"
os.environ["R2_SECRET_ACCESS_KEY"] = "cee49fead9c1a8ac2741a4c2703c908efc5d965100a2d8d20c233fce05547a55"
os.environ["R2_BUCKET"] = "sala-2026-hackathon-data"

print(f"✓ R2 credentials configured")
print(f"  Endpoint: {os.environ.get('R2_ENDPOINT', '')[:30]}...")
print(f"  Bucket: {os.environ.get('R2_BUCKET', '')}")

# === R2 Health Check (quick connectivity test, no dataset download) ===
import r2_download as hd

required = ["R2_ENDPOINT", "R2_ACCESS_KEY_ID", "R2_SECRET_ACCESS_KEY", "R2_BUCKET"]
missing = [k for k in required if not os.environ.get(k)]
if missing:
    raise ValueError(f"R2 credentials not ready for health check. Missing={missing}.")

client = hd.get_s3_client()
# Confirm API access against the bucket
bucket_probe = client.list_objects_v2(Bucket=os.environ["R2_BUCKET"], MaxKeys=1)
print("✓ Bucket access: OK")
print(f"  KeyCount (sample): {bucket_probe.get('KeyCount', 0)}")

# === Create local manifest with proper directory handling ===
manifest_path = Path.cwd() / "manifest.json"
print(f"\nManifest cache path: {manifest_path}")

# Download manifest from R2
resp = client.get_object(Bucket=os.environ["R2_BUCKET"], Key="manifest.json")
manifest = json.loads(resp["Body"].read().decode("utf-8"))

# Save manifest locally
manifest_path.parent.mkdir(parents=True, exist_ok=True)
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print(f"✓ Manifest created: {manifest_path}")
print(f"  Manifest entries: {len(manifest.get('files', []))}")
print(f"  File exists: {manifest_path.exists()}")

# === Download dataset using helper (precipitation-nowcasting) ===
print('\nDownloading precipitation-nowcasting dataset (this may take several minutes)...')
stats = hd.download_dataset(manifest, dataset_name='precipitation-nowcasting')
print(f"\nDownloaded: {stats.get('downloaded')}, Skipped: {stats.get('skipped')}, Failed: {stats.get('failed')}")


r2_download.py already present.
✓ R2 credentials configured
  Endpoint: https://6200702e94592ad231a53d...
  Bucket: sala-2026-hackathon-data
✓ Bucket access: OK
  KeyCount (sample): 1

Manifest cache path: /content/manifest.json
✓ Manifest created: /content/manifest.json
  Manifest entries: 0
  File exists: True



Shards: 100%|██████████| 14/14 [00:00<00:00, 998.03shard/s, file=MIRA_consolid_f15.csv, size_mb=79]     


Downloaded: 0, Skipped: 14, Failed: 0


## Parameters (replace as needed)

Set dataset and training parameters here instead of using argparse.

In [78]:
DATA_DIR = 'precipitation-nowcasting'  # o ruta absoluta
OUTPUT_DIR = 'checkpoints'
TARGET_STATION = 'jun'
LOOKBACK = 96
BATCH_SIZE = 128  # Reduced from 256 for better memory efficiency
EPOCHS = 50
PATIENCE = 10
LR = 1e-3
HIDDEN_DIM = 128
NUM_WORKERS = 4 if torch.cuda.is_available() else 0
PIN_MEMORY = torch.cuda.is_available()

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Parameters set:')
print(' DATA_DIR =', DATA_DIR)
print(' OUTPUT_DIR =', OUTPUT_DIR)
print(' TARGET_STATION =', TARGET_STATION)
print(' BATCH_SIZE =', BATCH_SIZE)
print(' NUM_WORKERS =', NUM_WORKERS)


Parameters set:
 DATA_DIR = precipitation-nowcasting
 OUTPUT_DIR = checkpoints
 TARGET_STATION = jun
 BATCH_SIZE = 128
 NUM_WORKERS = 0


In [79]:
# Cell: Verify data and directories exist
import os
from pathlib import Path

print('=== Local Execution Diagnostics ===\n')

# Check current working directory
cwd = Path.cwd()
print(f'Current directory: {cwd}')

# Check if DATA_DIR exists
data_path = Path(DATA_DIR)
print(f'\nDATA_DIR path: {data_path.resolve()}')
print(f'  Exists: {data_path.exists()}')

# Check for weather_stations subfolder
stations_path = data_path / 'weather_stations'
print(f'\nweather_stations folder: {stations_path.resolve()}')
print(f'  Exists: {stations_path.exists()}')

if stations_path.exists():
    csvs = list(stations_path.glob('*.csv'))
    print(f'  CSV files found: {len(csvs)}')
    for csv in csvs:
        size_mb = csv.stat().st_size / (1024*1024)
        print(f'    ✓ {csv.name} ({size_mb:.1f} MB)')
else:
    print(f'\n⚠️  MISSING FOLDER: {stations_path}')
    print('\nTo fix this, you need to download the CSV files into that folder.')
    print('Option 1: Run the R2 download cell above')
    print('Option 2: Download manually and place CSVs in precipitation-nowcasting/weather_stations/')
    print('Option 3: Copy CSVs from an existing location:')
    print('   shutil.copytree("source_path/weather_stations", "precipitation-nowcasting/weather_stations")')

# Check OUTPUT_DIR
output_path = Path(OUTPUT_DIR)
print(f'\nOUTPUT_DIR path: {output_path.resolve()}')
print(f'  Exists: {output_path.exists()}')
if not output_path.exists():
    output_path.mkdir(parents=True, exist_ok=True)
    print(f'  Created: ✓')


=== Local Execution Diagnostics ===

Current directory: /content

DATA_DIR path: /content/precipitation-nowcasting
  Exists: True

weather_stations folder: /content/precipitation-nowcasting/weather_stations
  Exists: True
  CSV files found: 8
    ✓ MIRA_consolid_f15.csv (4.3 MB)
    ✓ MERC_consolid_f15.csv (4.3 MB)
    ✓ california_housing_train.csv (1.6 MB)
    ✓ mnist_test.csv (17.4 MB)
    ✓ mnist_train_small.csv (34.8 MB)
    ✓ JUN_consolid_f15.csv (4.3 MB)
    ✓ CER_consolid_f15.csv (4.3 MB)
    ✓ california_housing_test.csv (0.3 MB)

OUTPUT_DIR path: /content/checkpoints
  Exists: True


In [80]:
# Cell: Debug - Find CSV files anywhere in the system
import os
import glob
from pathlib import Path
from subprocess import run, PIPE

print('=== DEEP SEARCH FOR CSV FILES ===\n')

# Method 1: Using find command (Unix/Linux)
print('Method 1: Using find command...')
try:
    result = run(['find', '/', '-name', '*.csv', '-type', 'f'], 
                 stdout=PIPE, stderr=PIPE, timeout=30, text=True)
    csv_paths = [p for p in result.stdout.split('\n') if 'consolid' in p.lower()]
    if csv_paths:
        print(f'✓ Found {len(csv_paths)} precipitation CSV files:')
        for path in csv_paths[:10]:
            if path:
                print(f'  {path}')
        if len(csv_paths) > 10:
            print(f'  ... and {len(csv_paths) - 10} more')
    else:
        print('✗ No precipitation CSV files found with find')
except Exception as e:
    print(f'✗ find command failed: {e}')

# Method 2: Manual recursive search in common locations
print('\nMethod 2: Manual search in common directories...')
search_bases = ['/tmp', '/root', '/home', '/content', '.']
found_csvs = {}

for base in search_bases:
    if not os.path.isdir(base):
        continue
    try:
        for root, dirs, files in os.walk(base):
            for file in files:
                if file.endswith('.csv') and 'consolid' in file.lower():
                    full_path = os.path.join(root, file)
                    found_csvs[file] = full_path
                    print(f'  ✓ Found: {full_path}')
    except PermissionError:
        pass
    except Exception as e:
        pass

if found_csvs:
    print(f'\n✓ Total CSV files found: {len(found_csvs)}')
    # Show the directory containing the CSVs
    first_csv = list(found_csvs.values())[0]
    source_dir = str(Path(first_csv).parent)
    print(f'\nSource directory containing CSVs: {source_dir}')
    print('\nContents:')
    try:
        for item in os.listdir(source_dir):
            item_path = os.path.join(source_dir, item)
            if os.path.isfile(item_path):
                size_mb = os.path.getsize(item_path) / (1024*1024)
                print(f'  📄 {item} ({size_mb:.1f} MB)')
    except Exception as e:
        print(f'  Error listing: {e}')
else:
    print('✗ No CSV files found anywhere')

# Method 3: Check what was actually downloaded
print('\n' + '='*60)
print('Checking /tmp for precipitation-nowcasting directory...')
for item in os.listdir('/tmp')[:50]:
    if 'precip' in item.lower() or 'weather' in item.lower():
        item_path = os.path.join('/tmp', item)
        if os.path.isdir(item_path):
            print(f'  📁 /tmp/{item}/')
            try:
                for subitem in os.listdir(item_path)[:10]:
                    subitem_path = os.path.join(item_path, subitem)
                    if os.path.isdir(subitem_path):
                        print(f'     📁 {subitem}/')
                    else:
                        print(f'     📄 {subitem}')
            except:
                pass

=== DEEP SEARCH FOR CSV FILES ===

Method 1: Using find command...
✓ Found 8 precipitation CSV files:
  /content/hackathon_data/precipitation-nowcasting/weather_stations/MIRA_consolid_f15.csv
  /content/hackathon_data/precipitation-nowcasting/weather_stations/MERC_consolid_f15.csv
  /content/hackathon_data/precipitation-nowcasting/weather_stations/JUN_consolid_f15.csv
  /content/hackathon_data/precipitation-nowcasting/weather_stations/CER_consolid_f15.csv
  /content/precipitation-nowcasting/weather_stations/MIRA_consolid_f15.csv
  /content/precipitation-nowcasting/weather_stations/MERC_consolid_f15.csv
  /content/precipitation-nowcasting/weather_stations/JUN_consolid_f15.csv
  /content/precipitation-nowcasting/weather_stations/CER_consolid_f15.csv

Method 2: Manual search in common directories...
  ✓ Found: /content/hackathon_data/precipitation-nowcasting/weather_stations/MIRA_consolid_f15.csv
  ✓ Found: /content/hackathon_data/precipitation-nowcasting/weather_stations/MERC_consolid_f1

## Load Script Code (functions and classes)

Below we define the functions, classes and logic from the original script. Cells are grouped: constants, data loaders, preprocessing, dataset, models, training/eval utilities.

In [81]:
# Cell: Constants and configuration
STATION_FILES = {
    'cer': 'weather_stations/CER_consolid_f15.csv',
    'jun': 'weather_stations/JUN_consolid_f15.csv',
    'merc': 'weather_stations/MERC_consolid_f15.csv',
    'mira': 'weather_stations/MIRA_consolid_f15.csv',
}

COLUMN_MAP = {
    'rain_mm': ['Rain_mm_Tot'],
    'temp_c': ['AirTC_Avg'],
    'rh_avg': ['RH_Avg'],
    'rh_max': ['RH_Max'],
    'rh_min': ['RH_Min'],
    'solar_kw': ['SlrkW_Avg'],
    'net_rad_wm2': ['NR_Wm2_Avg'],
    'wind_speed_ms': ['WS_ms_Avg'],
    'wind_dir': ['WindDir'],
    'soil_moisture_1': ['VW_Avg', 'VW'],
    'soil_moisture_2': ['VW_2_Avg', 'VW_2'],
    'soil_moisture_3': ['VW_3_Avg', 'VW_3'],
    'leaf_wetness': ['LWmV_Avg'],
    'leaf_wet_minutes': ['LWMWet_Tot'],
}

HORIZONS = {'3h': 12, '6h': 24, '12h': 48}
SEED = 42

print('Constants loaded')


Constants loaded


In [88]:
# Cell: Data loading functions
from datetime import timedelta

def load_station(name, data_dir):
    path = os.path.join(data_dir, STATION_FILES[name])
    df = pd.read_csv(path, low_memory=False)
    # Handle different timestamp formats
    try:
        df['TIMESTAMP'] = pd.to_datetime(df['TIMESTAMP'], format='%m/%d/%Y %H:%M')
    except ValueError:
        try:
            df['TIMESTAMP'] = pd.to_datetime(df['TIMESTAMP'], format='ISO8601')
        except ValueError:
            # Fall back to automatic parsing
            df['TIMESTAMP'] = pd.to_datetime(df['TIMESTAMP'])
    df = df.set_index('TIMESTAMP').sort_index()
    rename = {}
    for harmonized, candidates in COLUMN_MAP.items():
        for candidate in candidates:
            if candidate in df.columns:
                rename[candidate] = harmonized
                break
    df = df[list(rename.keys())].rename(columns=rename)
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df


def load_all_stations(data_dir):
    stations = {}
    for name in STATION_FILES:
        print(f'Loading {name}...')
        stations[name] = load_station(name, data_dir)
        df = stations[name]
        missing = [c for c in COLUMN_MAP if c not in df.columns]
        print(f"  {name:8s}: {df.index.min().date()} -> {df.index.max().date()} ({df.shape[0]:,} rows, {len(df.columns)} cols)" + (f"  missing={missing}" if missing else ""))
    return stations

print('Data loader functions defined')


Data loader functions defined


In [83]:
# Cell: Preprocessing & feature engineering

def merge_stations(stations):
    prefixed = []
    for name, stn_df in stations.items():
        prefixed.append(stn_df.add_prefix(f'{name}_'))
    df = pd.concat(prefixed, axis=1, sort=True)
    all_nan_cols = [c for c in df.columns if df[c].isnull().all()]
    if all_nan_cols:
        print(f"  Dropping {len(all_nan_cols)} entirely-NaN columns: {all_nan_cols}")
        df = df.drop(columns=all_nan_cols)
    return df


def impute(df, station_names):
    for stn in station_names:
        for var in ['temp_c', 'rh_avg', 'rh_max', 'rh_min', 'solar_kw',
                    'net_rad_wm2', 'soil_moisture_1', 'soil_moisture_2',
                    'soil_moisture_3']:
            col = f'{stn}_{var}'
            if col in df.columns:
                df[col] = df[col].interpolate(method='time', limit=24)
        col = f'{stn}_wind_speed_ms'
        if col in df.columns:
            df[col] = df[col].ffill(limit=4)
            df[col] = df[col].interpolate(method='time', limit=8)
        col = f'{stn}_wind_dir'
        if col in df.columns:
            df[col] = df[col].ffill(limit=8)
        for var in ['leaf_wetness', 'leaf_wet_minutes']:
            col = f'{stn}_{var}'
            if col in df.columns:
                df[col] = df[col].ffill(limit=8)
        rain_col = f'{stn}_rain_mm'
        if rain_col in df.columns:
            df[f'{stn}_rain_missing'] = df[rain_col].isnull().astype(float)
            df[rain_col] = df[rain_col].fillna(0.0)

    df = df.ffill(limit=96).bfill(limit=96)
    still_nan = df.isnull().sum().sum()
    if still_nan > 0:
        n_cols = (df.isnull().sum() > 0).sum()
        print(f"  Filling {still_nan:,} remaining NaN across {n_cols} columns with 0")
    df = df.fillna(0.0)
    df = df.copy()
    return df


def engineer_features(df, station_names):
    df['hour_sin'] = np.sin(2 * np.pi * df.index.hour / 24)
    df['hour_cos'] = np.cos(2 * np.pi * df.index.hour / 24)
    df['doy_sin'] = np.sin(2 * np.pi * df.index.dayofyear / 365.25)
    df['doy_cos'] = np.cos(2 * np.pi * df.index.dayofyear / 365.25)
    for stn in station_names:
        wd_col = f'{stn}_wind_dir'
        ws_col = f'{stn}_wind_speed_ms'
        temp_col = f'{stn}_temp_c'
        rh_col = f'{stn}_rh_avg'
        if wd_col in df.columns and ws_col in df.columns:
            wd_rad = np.deg2rad(df[wd_col])
            df[f'{stn}_wind_x'] = df[ws_col] * np.cos(wd_rad)
            df[f'{stn}_wind_y'] = df[ws_col] * np.sin(wd_rad)
        if temp_col in df.columns and rh_col in df.columns:
            T = df[temp_col]
            RH = df[rh_col].clip(lower=1)
            alpha = (17.27 * T) / (237.3 + T) + np.log(RH / 100)
            df[f'{stn}_dewpoint'] = (237.3 * alpha) / (17.27 - alpha)
            df[f'{stn}_dewpoint_depression'] = T - df[f'{stn}_dewpoint']
        sm_col = f'{stn}_soil_moisture_1'
        if sm_col in df.columns:
            df[f'{stn}_soil_moist_tend_3h'] = df[sm_col].diff(periods=12)
        for window, wlabel in [(4, '1h'), (12, '3h'), (24, '6h')]:
            rain_col = f'{stn}_rain_mm'
            if rain_col in df.columns:
                df[f'{stn}_rain_sum_{wlabel}'] = df[rain_col].rolling(window, min_periods=1).sum()
            if temp_col in df.columns:
                df[f'{stn}_temp_mean_{wlabel}'] = df[temp_col].rolling(window, min_periods=1).mean()
                df[f'{stn}_temp_std_{wlabel}'] = df[temp_col].rolling(window, min_periods=1).std()
            if ws_col in df.columns:
                df[f'{stn}_wind_mean_{wlabel}'] = df[ws_col].rolling(window, min_periods=1).mean()
            if rh_col in df.columns:
                df[f'{stn}_rh_mean_{wlabel}'] = df[rh_col].rolling(window, min_periods=1).mean()
    return df


def create_labels(df, station_names, target_station):
    for stn in station_names:
        rain_col = f'{stn}_rain_mm'
        for label, steps in HORIZONS.items():
            df[f'rain_future_{label}_{stn}'] = (
                df[rain_col].rolling(steps, min_periods=1).sum().shift(-steps)
            )
    train_mask = df.index < '2023-01-01'
    thresholds = {}
    for label, steps in HORIZONS.items():
        col = f'rain_future_{label}_{target_station}'
        wet = df.loc[train_mask, col]
        wet = wet[wet > 0]
        p95 = wet.quantile(0.95)
        thresholds[label] = max(p95, 2.0)
        print(f"  {label}: p95={p95:.2f}mm, threshold={thresholds[label]:.2f}mm")
    for stn in station_names:
        for label in HORIZONS:
            col = f'rain_future_{label}_{stn}'
            df[f'heavy_rain_{label}_{stn}'] = (df[col] >= thresholds[label]).astype(float)
            df.loc[df[col].isnull(), f'heavy_rain_{label}_{stn}'] = np.nan
    for label in HORIZONS:
        df[f'heavy_rain_{label}'] = df[f'heavy_rain_{label}_{target_station}']
    temp_col = f'{target_station}_temp_c'
    df['day_of_year'] = df.index.dayofyear
    daily_clim = df.loc[train_mask].groupby('day_of_year')[temp_col].agg(['mean', 'std'])
    daily_clim['mean_smooth'] = daily_clim['mean'].rolling(15, center=True, min_periods=5).mean()
    daily_clim['std_smooth'] = daily_clim['std'].rolling(15, center=True, min_periods=5).mean()
    daily_clim = daily_clim.bfill().ffill()
    df['temp_clim_mean'] = df['day_of_year'].map(daily_clim['mean_smooth'])
    df['temp_clim_std'] = df['day_of_year'].map(daily_clim['std_smooth'])
    df['temp_anomaly'] = (df[temp_col] - df['temp_clim_mean']) / df['temp_clim_std']
    df['temp_extreme'] = (df['temp_anomaly'].abs() > 2).astype(float)
    return df, thresholds

print('Preprocessing functions defined')


Preprocessing functions defined


In [84]:
# Cell: Dataset class
class WeatherDataset(Dataset):
    def __init__(self, df, feature_cols, target_col, lookback=96):
        self.lookback = lookback
        valid_mask = df[feature_cols + [target_col]].notna().all(axis=1)
        clean_df = df.loc[valid_mask].copy()
        self.features = clean_df[feature_cols].values.astype(np.float32)
        self.labels = clean_df[target_col].values.astype(np.float32)
        self.timestamps = clean_df.index
        self.valid_indices = []
        expected_delta = pd.Timedelta(minutes=15)
        for i in tqdm(range(lookback, len(self.features)), desc=f"Building windows ({target_col})", leave=False):
            window_times = self.timestamps[i - lookback:i + 1]
            diffs = window_times[1:] - window_times[:-1]
            if (diffs == expected_delta).all():
                self.valid_indices.append(i)
        self.valid_indices = np.array(self.valid_indices)
        print(f"  {target_col}: {len(self.valid_indices):,} valid windows from {len(self.features):,} rows (positive rate: {self.labels[self.valid_indices].mean():.3%})")
    def __len__(self):
        return len(self.valid_indices)
    def __getitem__(self, idx):
        i = self.valid_indices[idx]
        x = self.features[i - self.lookback:i]
        y = self.labels[i]
        return torch.from_numpy(x), torch.tensor(y)

print('Dataset class defined')


Dataset class defined


In [85]:
# Cell: Model definitions (RNN/LSTM/GRU + Attention + LNN optional)
class RecurrentClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout=0.3, rnn_type='lstm'):
        super().__init__()
        self.rnn_type = rnn_type
        rnn_cls = {'rnn': nn.RNN, 'lstm': nn.LSTM, 'gru': nn.GRU}[rnn_type]
        self.rnn = rnn_cls(input_size=input_dim, hidden_size=hidden_dim, num_layers=num_layers, dropout=dropout if num_layers>1 else 0, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout), 
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.ReLU(),
            nn.Dropout(dropout//2),
            nn.Linear(hidden_dim//2, 1)
        )
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights for better training stability."""
        for name, param in self.named_parameters():
            if 'weight_ih' in name or 'weight_hh' in name:
                nn.init.orthogonal_(param)
            elif 'bias' in name:
                nn.init.constant_(param, 0)
            elif 'weight' in name and 'classifier' in name:
                nn.init.xavier_uniform_(param)
    
    def forward(self, x):
        output, _ = self.rnn(x)
        last_hidden = output[:, -1, :]
        logit = self.classifier(last_hidden)
        return logit.squeeze(-1)


class MultiHeadAttention(nn.Module):
    """Multi-head attention mechanism."""
    def __init__(self, hidden_dim, num_heads=4):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        assert hidden_dim % num_heads == 0, "hidden_dim must be divisible by num_heads"
        
        self.head_dim = hidden_dim // num_heads
        self.scale = np.sqrt(self.head_dim)
        
        self.query = nn.Linear(hidden_dim, hidden_dim)
        self.key = nn.Linear(hidden_dim, hidden_dim)
        self.value = nn.Linear(hidden_dim, hidden_dim)
        self.fc_out = nn.Linear(hidden_dim, hidden_dim)
        
    def forward(self, values, keys, query, mask=None):
        """
        Args:
            values: (batch_size, seq_len, hidden_dim)
            keys: (batch_size, seq_len, hidden_dim)
            query: (batch_size, seq_len, hidden_dim)
            mask: optional mask tensor
        """
        batch_size = query.shape[0]
        
        # Linear transformations
        Q = self.query(query)
        K = self.key(keys)
        V = self.value(values)
        
        # Reshape for multi-head attention
        Q = Q.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attention_weights = torch.softmax(scores, dim=-1)
        context = torch.matmul(attention_weights, V)
        
        # Concatenate heads
        context = context.transpose(1, 2).contiguous()
        context = context.view(batch_size, -1, self.hidden_dim)
        
        output = self.fc_out(context)
        return output, attention_weights


class LSTMAttentionClassifier(nn.Module):
    """LSTM with Multi-Head Attention mechanism."""
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout=0.3, num_heads=4):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.lstm = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim, 
                            num_layers=num_layers, dropout=dropout if num_layers>1 else 0, 
                            batch_first=True)
        self.attention = MultiHeadAttention(hidden_dim, num_heads=num_heads)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.ReLU(),
            nn.Dropout(dropout//2),
            nn.Linear(hidden_dim//2, 1)
        )
        self._init_weights()
    
    def _init_weights(self):
        """Initialize weights for better training stability."""
        for name, param in self.named_parameters():
            if 'weight_ih' in name or 'weight_hh' in name:
                nn.init.orthogonal_(param)
            elif 'bias' in name:
                nn.init.constant_(param, 0)
            elif 'weight' in name and ('classifier' in name or 'fc_out' in name):
                nn.init.xavier_uniform_(param)
    
    def forward(self, x):
        """
        Args:
            x: (batch_size, seq_len, input_dim)
        Returns:
            logit: (batch_size,)
        """
        # LSTM encoding
        lstm_out, (h_n, c_n) = self.lstm(x)  # lstm_out: (batch_size, seq_len, hidden_dim)
        
        # Multi-head attention
        # Use lstm_out as query, keys, and values for self-attention
        attn_out, attn_weights = self.attention(lstm_out, lstm_out, lstm_out)
        
        # Combine LSTM and attention outputs (residual connection)
        combined = lstm_out + attn_out
        combined = self.dropout(combined)
        
        # Use last time step for classification
        last_hidden = combined[:, -1, :]
        logit = self.classifier(last_hidden)
        return logit.squeeze(-1)


# Check ncps availability at module level
try:
    from ncps.torch import LTC
    HAS_NCPS = True
except Exception:
    HAS_NCPS = False
    LTC = None

class LiquidNeuralNetworkClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, dropout=0.3):
        super().__init__()
        if not HAS_NCPS or LTC is None:
            raise RuntimeError('ncps library not available; install with: pip install ncps')
        self.ltc = LTC(input_dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(nn.Linear(hidden_dim, hidden_dim//2), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_dim//2, 1))
    def forward(self, x):
        hidden_states = self.ltc(x)
        final_hidden = hidden_states[:, -1, :]
        final_hidden = self.dropout(final_hidden)
        logit = self.classifier(final_hidden)
        return logit.squeeze(-1)

print('Model classes defined (including LSTM+Attention)')


Model classes defined


In [92]:
# Cell: Training utilities

def compute_class_weight(dataset):
    labels = dataset.labels[dataset.valid_indices]
    pos_rate = labels.mean()
    if pos_rate == 0 or pos_rate == 1:
        return 1.0
    return min((1 - pos_rate) / pos_rate, 20.0)


def train_model(model, train_loader, val_loader, pos_weight, device, lr=1e-3, max_epochs=50, patience=10, model_name='Model'):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight).to(device))
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
    best_val_loss = float('inf')
    best_state = None
    epochs_no_improve = 0
    history = {'train_loss': [], 'val_loss': []}
    pbar = tqdm(range(max_epochs), desc=f"Training {model_name}")
    for epoch in pbar:
        model.train()
        train_losses = []
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            logits = model(x_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(loss.item())
        model.eval()
        val_losses = []
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                logits = model(x_batch)
                loss = criterion(logits, y_batch)
                val_losses.append(loss.item())
        train_loss = np.mean(train_losses)
        val_loss = np.mean(val_losses)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        scheduler.step(val_loss)
        pbar.set_postfix(train=f"{train_loss:.4f}", val=f"{val_loss:.4f}", lr=f"{optimizer.param_groups[0]['lr']:.1e}")
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"  Early stopping at epoch {epoch + 1}")
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return history


def get_predictions(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(device)
            logits = model(x_batch)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_probs.append(probs)
            all_labels.append(y_batch.numpy())
    return np.concatenate(all_probs), np.concatenate(all_labels)


def platt_scale(val_probs, val_labels, test_probs):
    val_logits = np.log(val_probs / (1 - np.clip(val_probs, 1e-9, 1-1e-9)) + 1e-9).reshape(-1, 1)
    test_logits = np.log(test_probs / (1 - np.clip(test_probs, 1e-9, 1-1e-9)) + 1e-9).reshape(-1, 1)
    lr = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000)
    lr.fit(val_logits, val_labels)
    return lr.predict_proba(test_logits)[:, 1]


def evaluate_model(y_true, y_prob, model_name='Model', threshold=None):
    if threshold is None:
        prec, rec, thresholds = precision_recall_curve(y_true, y_prob)
        f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
        threshold = thresholds[np.argmax(f1)]
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    pod = tp / (tp + fn) if (tp + fn) > 0 else 0
    far = fp / (tp + fp) if (tp + fp) > 0 else 0
    csi = tp / (tp + fn + fp) if (tp + fn + fp) > 0 else 0
    f1 = 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) > 0 else 0
    pr_auc = average_precision_score(y_true, y_prob)
    roc_auc = roc_auc_score(y_true, y_prob) if y_true.sum() > 0 else 0
    brier = brier_score_loss(y_true, y_prob)
    clim_rate = y_true.mean()
    brier_clim = clim_rate * (1 - clim_rate)
    bss = 1 - brier / brier_clim if brier_clim > 0 else 0
    return {
        'model': model_name, 'threshold': threshold,
        'PR-AUC': pr_auc, 'ROC-AUC': roc_auc, 'BSS': bss,
        'CSI': csi, 'POD': pod, 'FAR': far, 'F1': f1,
        'TP': int(tp), 'FP': int(fp), 'FN': int(fn), 'TN': int(tn),
        'N_events': int(y_true.sum()),
    }


def save_eval_plots(y_true, y_prob, model_name, output_dir):
    """Generate and save evaluation plots (PR curve, calibration, histogram, confusion matrix)."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # PR curve
    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    axes[0, 0].plot(rec, prec, linewidth=2, color='blue')
    axes[0, 0].fill_between(rec, prec, alpha=0.2, color='blue')
    axes[0, 0].set_xlabel('Recall')
    axes[0, 0].set_ylabel('Precision')
    axes[0, 0].set_title(f'{model_name} — Precision-Recall Curve')
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].set_xlim([0, 1])
    axes[0, 0].set_ylim([0, 1])
    
    # Calibration plot
    frac_pos, mean_pred = calibration_curve(y_true, y_prob, n_bins=10)
    axes[0, 1].plot([0, 1], [0, 1], 'k--', label='Perfect calibration', linewidth=2)
    axes[0, 1].plot(mean_pred, frac_pos, 'o-', linewidth=2, markersize=8, label=model_name, color='red')
    axes[0, 1].set_xlabel('Mean Predicted Probability')
    axes[0, 1].set_ylabel('Fraction of Positives')
    axes[0, 1].set_title(f'{model_name} — Calibration Curve')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].set_xlim([0, 1])
    axes[0, 1].set_ylim([0, 1])
    
    # Histogram of predictions
    axes[1, 0].hist(y_prob[y_true == 0], bins=30, alpha=0.6, label='Negative', color='blue')
    axes[1, 0].hist(y_prob[y_true == 1], bins=30, alpha=0.6, label='Positive', color='red')
    axes[1, 0].set_xlabel('Predicted Probability')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].set_title(f'{model_name} — Prediction Distribution')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Confusion matrix
    threshold = 0.5
    y_pred = (y_prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    im = axes[1, 1].imshow(cm, cmap='Blues', aspect='auto')
    axes[1, 1].set_xlabel('Predicted')
    axes[1, 1].set_ylabel('Actual')
    axes[1, 1].set_title(f'{model_name} — Confusion Matrix (threshold={threshold})')
    axes[1, 1].set_xticks([0, 1])
    axes[1, 1].set_yticks([0, 1])
    axes[1, 1].set_xticklabels(['No Rain', 'Rain'])
    axes[1, 1].set_yticklabels(['No Rain', 'Rain'])
    for i in range(2):
        for j in range(2):
            color = 'white' if cm[i, j] > cm.max()/2 else 'black'
            axes[1, 1].text(j, i, str(cm[i, j]), ha='center', va='center', color=color, fontsize=12, fontweight='bold')
    plt.colorbar(im, ax=axes[1, 1])
    
    plt.tight_layout()
    fig_path = os.path.join(output_dir, f'{model_name.lower()}_eval_plots.png')
    plt.savefig(fig_path, dpi=100, bbox_inches='tight')
    plt.close()
    print(f"  Saved {model_name} evaluation plots -> {fig_path}")


print('Training utilities defined')


Training utilities defined


In [ ]:
# Cell: Run complete pipeline (load, preprocess, train ALL models, evaluate & compare)
print('\n' + '='*80)
print('COMPLETE PIPELINE: Data Loading → Training → Evaluation → Comparison')
print('='*80)

print('\n=== Stage 1: Loading data ===')
try:
    stations = load_all_stations(DATA_DIR)
except FileNotFoundError as e:
    print(f'\n❌ ERROR: Data files not found!')
    print(f'   Expected location: {os.path.abspath(DATA_DIR)}')
    raise

print('\n=== Stage 2: Merge & impute ===')
df = merge_stations(stations)
print(f'  Wide DataFrame: {df.shape[0]:,} rows x {df.shape[1]} cols')
df = impute(df, list(stations.keys()))

print('\n=== Stage 3: Feature engineering ===')
df = engineer_features(df, list(stations.keys()))
print(f'  Features: {df.shape[1]} columns')

print(f"\n=== Stage 4: Labels (target: {TARGET_STATION}) ===")
df, thresholds = create_labels(df, list(stations.keys()), TARGET_STATION)
for label in HORIZONS:
    col = f'heavy_rain_{label}'
    print(f'  {col}: {df[col].mean():.3%} positive ({df[col].sum():.0f} events)')

print('\n=== Stage 5: Split data (60/20/20) ===')
n_rows = len(df)
train_size = int(0.6 * n_rows)
val_size = int(0.2 * n_rows)
train_end_idx = train_size
val_end_idx = train_size + val_size

train_df = df.iloc[:train_end_idx].copy()
val_df = df.iloc[train_end_idx:val_end_idx].copy()
test_df = df.iloc[val_end_idx:].copy()

print(f"  Train: {train_df.index.min().date()} → {train_df.index.max().date()} ({len(train_df):,} rows)")
print(f"  Val:   {val_df.index.min().date()} → {val_df.index.max().date()} ({len(val_df):,} rows)")
print(f"  Test:  {test_df.index.min().date()} → {test_df.index.max().date()} ({len(test_df):,} rows)")

print('\n=== Stage 6: Normalize ===')
LABEL_COLS = [f'heavy_rain_{h}' for h in HORIZONS]
for stn in stations:
    LABEL_COLS += [f'heavy_rain_{h}_{stn}' for h in HORIZONS]
LABEL_COLS += ['temp_extreme', 'temp_anomaly']
FEATURE_COLS = [c for c in df.columns if c not in LABEL_COLS]
print(f'  Feature columns: {len(FEATURE_COLS)}')

train_mean = train_df[FEATURE_COLS].mean()
train_std = train_df[FEATURE_COLS].std().replace(0, 1)
for split_df in [train_df, val_df, test_df]:
    split_df[FEATURE_COLS] = (split_df[FEATURE_COLS] - train_mean) / train_std

print('\n=== Stage 7: Create datasets ===')
TARGET_COL = 'heavy_rain_3h'
train_ds = WeatherDataset(train_df, FEATURE_COLS, TARGET_COL, LOOKBACK)
val_ds = WeatherDataset(val_df, FEATURE_COLS, TARGET_COL, LOOKBACK)
test_ds = WeatherDataset(test_df, FEATURE_COLS, TARGET_COL, LOOKBACK)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
print(f'  Batches: train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)}')

print('\n=== Stage 8: Initialize ALL models ===')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'  GPU Memory: {torch.cuda.get_device_properties(device).total_memory / 1e9:.1f} GB')

n_features = len(FEATURE_COLS)
pos_weight = compute_class_weight(train_ds)
print(f'  Positive class weight: {pos_weight:.1f}x')

# Initialize all models
models = {}
model_configs = [
    ('LSTM', lambda: RecurrentClassifier(n_features, hidden_dim=HIDDEN_DIM, rnn_type='lstm')),
    ('LSTM+Attention', lambda: LSTMAttentionClassifier(n_features, hidden_dim=HIDDEN_DIM, num_heads=4)),
    ('GRU', lambda: RecurrentClassifier(n_features, hidden_dim=HIDDEN_DIM, rnn_type='gru')),
    ('RNN', lambda: RecurrentClassifier(n_features, hidden_dim=HIDDEN_DIM, rnn_type='rnn')),
]

print('\nInitializing models:')
for model_name, model_fn in model_configs:
    try:
        model = model_fn().to(device)
        models[model_name] = model
        n_params = sum(p.numel() for p in model.parameters())
        print(f'  ✓ {model_name:15s} ({n_params:,} params)')
    except Exception as e:
        print(f'  ⚠ {model_name:15s} skipped: {e}')

# Try LNN
if HAS_NCPS and LTC is not None:
    try:
        lnn = LiquidNeuralNetworkClassifier(n_features, hidden_dim=HIDDEN_DIM).to(device)
        models['LNN'] = lnn
        n_params = sum(p.numel() for p in lnn.parameters())
        print(f'  ✓ LNN             ({n_params:,} params)')
    except Exception as e:
        print(f'  ⚠ LNN skipped: {e}')

print(f'\n  Total models ready: {len(models)}')

print('\n' + '='*80)
print('TRAINING PHASE - All Models')
print('='*80)
histories = {}
for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"  Training {name}")
    print(f"{'='*50}")
    histories[name] = train_model(model, train_loader, val_loader, pos_weight=pos_weight, 
                                   device=device, lr=LR, max_epochs=EPOCHS, patience=PATIENCE, 
                                   model_name=name)

# Save checkpoints
print('\n' + '='*80)
print('CHECKPOINTS - Saving trained models')
print('='*80)
for name, model in models.items():
    ckpt_path = os.path.join(OUTPUT_DIR, f"{name.lower().replace('+', '_')}_heavy_rain_3h.pt")
    torch.save({
        'model_state_dict': model.state_dict(), 
        'model_type': name,
        'input_dim': n_features, 
        'hidden_dim': HIDDEN_DIM, 
        'target': TARGET_COL, 
        'target_station': TARGET_STATION, 
        'lookback': LOOKBACK, 
        'feature_cols': FEATURE_COLS, 
        'train_mean': train_mean.to_dict(), 
        'train_std': train_std.to_dict(), 
        'thresholds': thresholds,
    }, ckpt_path)
    print(f'  ✓ {name:15s} → {ckpt_path}')

# Training curves comparison
print('\n' + '='*80)
print('TRAINING CURVES - Loss over epochs')
print('='*80)
fig, axes = plt.subplots(2, (len(histories)+1)//2, figsize=(5*((len(histories)+1)//2), 8))
axes = axes.flatten() if len(histories) > 1 else [axes]
for ax, (name, hist) in zip(axes, histories.items()):
    ax.plot(hist['train_loss'], label='Train', linewidth=2)
    ax.plot(hist['val_loss'], label='Val', linewidth=2)
    ax.set_title(f'{name}', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)
for ax in axes[len(histories):]:
    ax.axis('off')
plt.tight_layout()
curves_path = os.path.join(OUTPUT_DIR, 'training_curves_all_models.png')
plt.savefig(curves_path, dpi=100, bbox_inches='tight')
plt.close()
print(f'✓ Saved training curves → {curves_path}')

print('\n' + '='*80)
print('EVALUATION PHASE - All Models on Test Set')
print('='*80)
results = []
for name, model in models.items():
    print(f'\n  Evaluating {name}...')
    val_probs, val_labels = get_predictions(model, val_loader, device)
    test_probs_raw, test_labels = get_predictions(model, test_loader, device)
    test_probs = platt_scale(val_probs, val_labels, test_probs_raw)
    val_probs_cal = platt_scale(val_probs, val_labels, val_probs)
    prec, rec, pr_thresholds = precision_recall_curve(val_labels, val_probs_cal)
    f1 = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
    best_thresh = pr_thresholds[np.argmax(f1)]
    metrics = evaluate_model(test_labels, test_probs, model_name=name, threshold=best_thresh)
    results.append(metrics)
    save_eval_plots(test_labels, test_probs, name, OUTPUT_DIR)

print('\n' + '='*80)
print('COMPARISON TABLE - Model Rankings')
print('='*80)
results_df = pd.DataFrame(results)
display_cols = ['model', 'PR-AUC', 'ROC-AUC', 'BSS', 'CSI', 'POD', 'FAR', 'F1', 'threshold']

# Sort by PR-AUC (best metric for imbalanced data)
results_df_sorted = results_df.sort_values('PR-AUC', ascending=False).reset_index(drop=True)
print('\n📊 RESULTS RANKED BY PR-AUC (Precision-Recall Area Under Curve)\n')
print(results_df_sorted[display_cols].to_string(index=True, float_format='%.4f'))

# Summary statistics
print('\n' + '='*80)
print('SUMMARY STATISTICS')
print('='*80)
summary_stats = results_df[['model', 'PR-AUC', 'ROC-AUC', 'BSS', 'CSI']].set_index('model')
print('\n' + summary_stats.to_string(float_format='%.4f'))

best_idx = results_df['PR-AUC'].idxmax()
best_model = results_df.loc[best_idx, 'model']
best_pr_auc = results_df.loc[best_idx, 'PR-AUC']
print(f'\n🏆 BEST MODEL: {best_model}')
print(f'   PR-AUC: {best_pr_auc:.4f}')

# Detailed comparison by metric
print('\n' + '='*80)
print('METRIC RANKINGS')
print('='*80)
metrics_to_rank = ['PR-AUC', 'ROC-AUC', 'CSI', 'POD', 'F1', 'BSS']
for metric in metrics_to_rank:
    best_val = results_df[metric].max()
    best_model_name = results_df[results_df[metric] == best_val]['model'].values[0]
    print(f'{metric:10s}: {best_model_name:15s} ({best_val:.4f})')

# Save detailed results
results_path = os.path.join(OUTPUT_DIR, 'results_comparison.json')
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.integer,)): return int(obj)
        if isinstance(obj, (np.floating,)): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        return super().default(obj)

with open(results_path, 'w') as f:
    json.dump(results, f, indent=2, cls=NumpyEncoder)
print(f'\n✓ Saved detailed results → {results_path}')

# Save comparison CSV
csv_path = os.path.join(OUTPUT_DIR, 'results_comparison.csv')
results_df_sorted[display_cols + ['N_events', 'TP', 'FP', 'FN', 'TN']].to_csv(csv_path, index=False)
print(f'✓ Saved comparison table → {csv_path}')

print('\n' + '='*80)
print('✨ PIPELINE COMPLETE - All models trained and compared')
print('='*80)



=== Stage 1: Loading data ===
Loading cer...
  cer     : 2021-01-01 -> 2021-07-19 (19,200 rows, 14 cols)
Loading jun...
  jun     : 2021-01-01 -> 2021-07-19 (19,200 rows, 14 cols)
Loading merc...
  merc    : 2021-01-01 -> 2021-07-19 (19,200 rows, 14 cols)
Loading mira...
  mira    : 2021-01-01 -> 2021-07-19 (19,200 rows, 14 cols)

=== Stage 2: Merge & impute ===
  Wide DataFrame: 19,200 rows x 56 cols
  After imputation: 19,200 rows x 60 cols

=== Stage 3: Feature engineering ===
  Features: 144 columns

=== Stage 4: Labels (target: jun) ===
  3h: p95=49.14mm, threshold=49.14mm
  6h: p95=69.61mm, threshold=69.61mm
  12h: p95=98.22mm, threshold=98.22mm
  heavy_rain_3h: 1.683% positive (323 events)
  heavy_rain_6h: 2.388% positive (458 events)
  heavy_rain_12h: 3.378% positive (647 events)

=== Stage 5: Split ===
  Train: 2021-01-01 -> 2021-04-30 (11,520 rows)
  Val:   2021-05-01 -> 2021-06-09 (3,840 rows)
  Test:  2021-06-10 -> 2021-07-19 (3,840 rows)

=== Stage 6: Normalize ===
  Feat

  heavy_rain_3h: 11,412 valid windows from 11,508 rows (positive rate: 1.306%)


  heavy_rain_3h: 3,744 valid windows from 3,840 rows (positive rate: 2.885%)


  heavy_rain_3h: 3,696 valid windows from 3,792 rows (positive rate: 1.461%)
  Batches: train=89, val=30, test=29

=== Stage 8: Train models ===
Using device: cpu
  Positive class weight: 20.0x
  ✓ Liquid Neural Network (LNN) added

  Training LSTM (288,385 params)


Training LSTM:   4%|▍         | 2/50 [02:19<55:51, 69.82s/it, lr=1.0e-03, train=0.0536, val=1.1296]

## Save & Download Results

After training you can download the `OUTPUT_DIR` or save it to Drive. The cell below provides examples.

In [ ]:
# Cell: Save results to Drive or download (examples)
# If Drive mounted, copy OUTPUT_DIR to Drive
try:
    from google.colab import files
    # Zip and download
    import shutil
    zip_path = '/content/checkpoints.zip'
    shutil.make_archive('/content/checkpoints', 'zip', OUTPUT_DIR)
    files.download(zip_path)
except Exception as e:
    print('files.download not available; if running locally, download OUTPUT_DIR manually or copy to Drive:', e)


## Troubleshooting

Common issues and fixes:

- Missing station files: ensure `DATA_DIR/weather_stations/` contains the CSV files from the R2 dataset.
- Memory/GPU OOM: reduce `BATCH_SIZE` and `LOOKBACK` or use a smaller model.
- Missing packages: re-run the install cell or install locally with pip.
- ncps not available: Liquid Neural Networks require `ncps`; run `pip install ncps` if needed (may need custom wheels).
- If training is very slow: try `EPOCHS=5` for a quick smoke test.

If you want, puedo añadir una celda con un ejemplo mínimo (toy dataset) para comprobar que las celdas funcionan sin datos reales.
